# Dimensionality-Preserving SSL - Analysis & Visualization

This notebook loads trained checkpoints and produces:
- Spectral analysis plots (eigenvalue decay, effective rank over training)
- Linear probe accuracy comparison
- Robustness evaluation on corruption benchmarks

Run this after training is complete.

In [ ]:
import os
if not os.path.exists("dim-ssl"):
    !git clone https://github.com/YOUR_GITHUB_USERNAME/dim-ssl.git
%cd dim-ssl
!pip install -e . -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models import SSLModel
from src.evaluation import compute_spectral_diagnostics
from src.data import get_eval_dataloaders

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
def load_model(ckpt_path, device="cuda"):
    """Load a trained model from checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg = ckpt["config"]
    model = SSLModel(
        backbone=cfg["backbone"],
        proj_hidden_dim=cfg["proj_hidden_dim"],
        proj_out_dim=cfg["proj_out_dim"],
        proj_layers=cfg["proj_layers"],
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model, cfg

# Load your checkpoints
# Adjust paths to your actual checkpoint locations
CKPT_DIR = "./checkpoints"  # or Google Drive path
# model_baseline, cfg_baseline = load_model(f"{CKPT_DIR}/simclr_baseline/epoch_199.pth")
# model_dimreg, cfg_dimreg = load_model(f"{CKPT_DIR}/simclr_dimreg/epoch_199.pth")

## Spectral Comparison: Baseline vs Dim-Reg

In [ ]:
def plot_spectral_comparison(models_dict, loader, device="cuda"):
    """
    Compare eigenvalue spectra of multiple models.
    models_dict: {"name": model, ...}
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for name, model in models_dict.items():
        diag = compute_spectral_diagnostics(model, loader, device=device)
        eigs = diag["eigenvalues"]
        
        # Eigenvalue spectrum (log scale)
        axes[0].semilogy(eigs, label=f"{name} (eff. rank={diag['effective_rank']:.1f})")
        
        # Cumulative variance
        cumvar = np.cumsum(eigs) / eigs.sum()
        axes[1].plot(cumvar, label=name)
        
        # Normalized eigenvalues (probability)
        p = eigs / eigs.sum()
        axes[2].semilogy(p, label=name)
    
    axes[0].set_xlabel("Component")
    axes[0].set_ylabel("Eigenvalue (log)")
    axes[0].set_title("Eigenvalue Spectrum")
    axes[0].legend()
    
    axes[1].set_xlabel("Component")
    axes[1].set_ylabel("Cumulative variance")
    axes[1].set_title("Cumulative Variance Explained")
    axes[1].axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
    axes[1].legend()
    
    axes[2].set_xlabel("Component")
    axes[2].set_ylabel("p_i (log)")
    axes[2].set_title("Normalized Spectrum")
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig("spectral_comparison.pdf", dpi=150, bbox_inches="tight")
    plt.show()

# Uncomment after loading models:
# _, eval_test = get_eval_dataloaders("cifar100", "./data")
# plot_spectral_comparison({"SimCLR": model_baseline, "SimCLR+DimReg": model_dimreg}, eval_test)

## Robustness: CIFAR-100-C Corruption Benchmark

In [ ]:
# CIFAR-100-C evaluation
# Download: https://zenodo.org/record/3555552
# Or generate with: https://github.com/hendrycks/robustness

CORRUPTIONS = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "glass_blur", "motion_blur", "zoom_blur",
    "snow", "frost", "fog", "brightness",
    "contrast", "elastic_transform", "pixelate", "jpeg_compression",
]
SEVERITIES = [1, 2, 3, 4, 5]

def evaluate_corruption_robustness(model, corruption_dir, device="cuda"):
    """Evaluate linear probe accuracy on each corruption type/severity."""
    # TODO: Implement after baseline training is working
    # 1. Load CIFAR-100-C numpy files
    # 2. Extract features with frozen encoder
    # 3. Apply pre-trained linear classifier
    # 4. Report per-corruption and mean corruption accuracy
    pass

print("Corruption evaluation - implement after baselines are trained")